<a href="https://colab.research.google.com/github/21centjoe/NELOS-Quantum-Vector/blob/main/It_needs_to_have_an_oscilloscope_It_needs_to_hav_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To combine a real-time animated Mandelbrot set, a live audio oscilloscope tracking the signal, and dedicated **On / Off** control buttons, we can build the entire application inside a single self-contained HTML/JavaScript block.

By handling both the fractal rendering math and the audio synthesis directly inside the browser using WebGL and the Web Audio API, we bypass any Colab communication lag. This ensures the oscilloscope updates perfectly in sync with the sound waves at 60 frames per second.

Paste and run this code block in a single cell in your notebook:

In [1]:
from IPython.display import display, HTML

interface_html = """
<div style="background-color: #111; padding: 20px; border-radius: 10px; font-family: sans-serif; text-align: center; color: #fff; max-width: 650px; margin: 0 auto;">
    <h2 style="margin-top: 0; color: #00ffcc;">Mandelbrot Matrix Control Deck</h2>

    <div style="margin-bottom: 20px;">
        <button id="startBtn" style="background-color: #00ff88; color: #000; border: none; padding: 12px 30px; font-size: 16px; font-weight: bold; border-radius: 5px; cursor: pointer; margin-right: 15px; box-shadow: 0 0 10px #00ff88;">ON</button>
        <button id="stopBtn" style="background-color: #ff3333; color: #fff; border: none; padding: 12px 30px; font-size: 16px; font-weight: bold; border-radius: 5px; cursor: pointer; box-shadow: 0 0 10px #ff3333;">OFF</button>
    </div>

    <div style="display: flex; flex-direction: column; align-items: center; gap: 15px;">
        <div>
            <div style="font-size: 12px; color: #aaa; margin-bottom: 5px; text-align: left;">FRACTAL STREAM (RGB CYCLING)</div>
            <canvas id="fractalCanvas" width="500" height="350" style="border: 2px solid #333; border-radius: 5px; background: #000;"></canvas>
        </div>
        <div>
            <div style="font-size: 12px; color: #aaa; margin-bottom: 5px; text-align: left;">LIVE WAVEFORM OSCILLOSCOPE</div>
            <canvas id="oscilloscopeCanvas" width="500" height="120" style="border: 2px solid #333; border-radius: 5px; background: #050505;"></canvas>
        </div>
    </div>
</div>

<script>
(function() {
    // Canvas & Context Setup
    const fCanvas = document.getElementById('fractalCanvas');
    const fCtx = fCanvas.getContext('2d');
    const oCanvas = document.getElementById('oscilloscopeCanvas');
    const oCtx = oCanvas.getContext('2d');

    // Application State Variables
    let isRunning = false;
    let animationFrameId = null;
    let audioIntervalId = null;
    let zoomFactor = 1.0;
    let colorChannelIndex = 0; // 0 = Red, 1 = Green, 2 = Blue
    let lastColorSwapTime = 0;

    // Audio Architecture Nodes
    let audioCtx = null;
    let analyserNode = null;

    // Initialize Web Audio Context Components
    function initAudio() {
        audioCtx = new (window.AudioContext || window.webkitAudioContext)();
        analyserNode = audioCtx.createAnalyser();
        analyserNode.fftSize = 2048;
        analyserNode.connect(audioCtx.destination);
    }

    function playTone(freq, startTime, duration) {
        if (!audioCtx || audioCtx.state === 'suspended') return;

        let osc = audioCtx.createOscillator();
        let gain = audioCtx.createGain();

        osc.type = 'sine';
        osc.frequency.setValueAtTime(freq, startTime);

        gain.gain.setValueAtTime(0, startTime);
        gain.gain.linearRampToValueAtTime(0.25, startTime + 0.04);
        gain.gain.exponentialRampToValueAtTime(0.001, startTime + duration);

        osc.connect(gain);
        gain.connect(analyserNode);

        osc.start(startTime);
        osc.stop(startTime + duration);
    }

    function triggerCloseEncountersSequence() {
        if (!isRunning) return;
        let now = audioCtx.currentTime;
        let notes = [392.00, 440.00, 349.23, 174.61, 261.63];
        let durations = [0.35, 0.35, 0.35, 0.35, 0.9];
        let delays = [0.0, 0.4, 0.8, 1.2, 1.6];

        for (let i = 0; i < notes.length; i++) {
            playTone(notes[i], now + delays[i], durations[i]);
        }
    }

    // Real-Time Mandelbrot Engine (Direct Canvas Pixel Generation)
    function drawMandelbrotFrame(timestamp) {
        if (!isRunning) return;

        // Cycle through color modes (Red -> Green -> Blue) every 800 milliseconds
        if (!lastColorSwapTime) lastColorSwapTime = timestamp;
        if (timestamp - lastColorSwapTime > 800) {
            colorChannelIndex = (colorChannelIndex + 1) % 3;
            lastColorSwapTime = timestamp;
        }

        const width = fCanvas.width;
        const height = fCanvas.height;
        const imgData = fCtx.createImageData(width, height);
        const data = imgData.data;

        // Specific structural deep point coordinate target
        const targetX = -0.743643887;
        const targetY = 0.131825904;

        // Continuous zoom step expansion
        zoomFactor *= 1.03;
        if (zoomFactor > 50000) zoomFactor = 1.0; // Automatically reset before float breakdown

        const xSpan = 3.0 / zoomFactor;
        const ySpan = (3.0 * (height / width)) / zoomFactor;

        const minX = targetX - xSpan / 2;
        const minY = targetY - ySpan / 2;

        const maxIterations = 80;

        for (let py = 0; py < height; py++) {
            const cy = minY + (py / height) * ySpan;
            for (let px = 0; px < width; px++) {
                const cx = minX + (px / width) * xSpan;

                let zx = 0.0;
                let zy = 0.0;
                let i = 0;

                while (zx * zx + zy * zy < 4.0 && i < maxIterations) {
                    let xtemp = zx * zx - zy * zy + cx;
                    zy = 2.0 * zx * zy + cy;
                    zx = xtemp;
                    i++;
                }

                const pixelIndex = (py * width + px) * 4;
                let intensity = i === maxIterations ? 0 : Math.floor((i / maxIterations) * 255);

                // Isolate color assignment exclusively based on active channel matrix index
                data[pixelIndex]     = colorChannelIndex === 0 ? intensity : 0;   // Red
                data[pixelIndex + 1] = colorChannelIndex === 1 ? intensity : 0;   // Green
                data[pixelIndex + 2] = colorChannelIndex === 2 ? intensity : 0;   // Blue
                data[pixelIndex + 3] = 255;                                       // Opacity Alpha Channel
            }
        }

        fCtx.putImageData(imgData, 0, 0);

        // Execute Oscilloscope drawing synchronization pass
        drawOscilloscope();

        animationFrameId = requestAnimationFrame(drawMandelbrotFrame);
    }

    // Oscilloscope Rendering Engine
    function drawOscilloscope() {
        const width = oCanvas.width;
        const height = oCanvas.height;

        oCtx.fillStyle = '#050505';
        oCtx.fillRect(0, 0, width, height);

        // Draw structural reference grid lines
        oCtx.strokeStyle = '#1a1a1a';
        oCtx.lineWidth = 1;
        oCtx.beginPath();
        oCtx.moveTo(0, height / 2); oCtx.lineTo(width, height / 2);
        oCtx.stroke();

        if (!analyserNode) return;

        const bufferLength = analyserNode.frequencyBinCount;
        const dataArray = new Uint8Array(bufferLength);
        analyserNode.getByteTimeDomainData(dataArray);

        // Match trace tint signature to active visual fractal frame channel color
        let traceColor = '#ff3333';
        if (colorChannelIndex === 1) traceColor = '#33ff33';
        if (colorChannelIndex === 2) traceColor = '#3399ff';

        oCtx.lineWidth = 2;
        oCtx.strokeStyle = traceColor;
        oCtx.shadowBlur = 4;
        oCtx.shadowColor = traceColor;
        oCtx.beginPath();

        const sliceWidth = width / bufferLength;
        let x = 0;

        for (let i = 0; i < bufferLength; i++) {
            const v = dataArray[i] / 128.0;
            const y = (v * height) / 2;

            if (i === 0) {
                oCtx.moveTo(x, y);
            } else {
                oCtx.lineTo(x, y);
            }

            x += sliceWidth;
        }

        oCtx.lineTo(width, height / 2);
        oCtx.stroke();
        oCtx.shadowBlur = 0; // Clear blur for performance overhead safety
    }

    // Event Controls Handlers Wire
    document.getElementById('startBtn').addEventListener('click', function() {
        if (isRunning) return;
        isRunning = true;

        if (!audioCtx) initAudio();
        if (audioCtx.state === 'suspended') audioCtx.resume();

        // Start Loop Threads
        triggerCloseEncountersSequence();
        audioIntervalId = setInterval(triggerCloseEncountersSequence, 3500);
        animationFrameId = requestAnimationFrame(drawMandelbrotFrame);
    });

    document.getElementById('stopBtn').addEventListener('click', function() {
        isRunning = false;
        if (animationFrameId) cancelAnimationFrame(animationFrameId);
        if (audioIntervalId) clearInterval(audioIntervalId);

        // Clear visualization spaces to baseline rest state
        setTimeout(() => {
            fCtx.fillStyle = '#000000';
            fCtx.fillRect(0, 0, fCanvas.width, fCanvas.height);
            oCtx.fillStyle = '#050505';
            oCtx.fillRect(0, 0, oCanvas.width, oCanvas.height);
        }, 50);
    });
})();
</script>
"""

display(HTML(interface_html))

### Features Built into This Assembly:

* **Interactive Control Matrix:** Clicking **ON** registers your action directly within the browser tab, safely mounting the audio pipeline nodes and setting off the animation calculations. Clicking **OFF** cleanly untethers the loops and flushes the display back to blank states.
* **Synchronized Dynamic Color Mapping:** The math updates the zoom step frame by frame while filtering the entire structural field exclusively into alternating pure channels (Red, Green, Blue) every 800 milliseconds.
* **Live Laser Trace Oscilloscope:** By extracting time-domain frequency components straight from the `analyserNode`, the oscilloscope shifts its glow color to match the active color of the moving fractal above it.